Imports

In [ ]:
import albumentations as A
import cv2
import numpy as np
import json
import numpy as np
import os
import sys
import torch
import torch.nn as nn
from insightface.app import FaceAnalysis
from sklearn.metrics.pairwise import cosine_similarity
from torchvision.models.video import mvit_v1_b

CUDA, Capture, & Import

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

vid_source=r'C:/Users/Dylan/Documents/School/Capstone/Footage.mp4'
cap = cv2.VideoCapture(vid_source)
total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
print("Source Video:", vid_source)

with open("C:/Users/Dylan/Documents/School/Capstone/tracking_output.json", "r") as f:
    tracking_data = json.load(f)

total_frames = tracking_data["total_frames"]
b_boxes = {int(k): v for k, v in tracking_data["b_boxes"].items()}
high_p_boxes = {int(k): v for k, v in tracking_data["high_conf_boxes"].items()}
p = {int(k): v for k, v in tracking_data["class_confidences"].items()}

InsightFace

In [ ]:
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider"])
app.prepare(ctx_id=0)

embeddings = {}

for track_id, (frame_number, b_box) in high_p_boxes.items():
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()
    if not ret:
        print(f"Failed to read frame {frame_number}")
        continue

    x1, y1, x2, y2 = b_box
    cropped_face = frame[y1:y2, x1:x2]

    faces = app.get(cropped_face)
    if faces:
        embeddings[track_id] = faces[0].embedding
    else:
        print(f"No face detected for track ID {track_id}")

cap.release()

similarity_threshold = 0.7 # Tune threshold for elimination
track_ids = list(embeddings.keys())
duplicates = set()

for i in range(len(track_ids)):
    for j in range(i + 1, len(track_ids)):
        id1, id2 = track_ids[i], track_ids[j]
        if id2 in duplicates:
            continue
        emb1 = embeddings[id1].reshape(1, -1)
        emb2 = embeddings[id2].reshape(1, -1)
        similarity = cosine_similarity(emb1, emb2)[0][0]

        if similarity > similarity_threshold:
            duplicates.add(id2)

for dup_id in duplicates:
    b_boxes.pop(dup_id, None)

Video MViT

In [ ]:
cap = cv2.VideoCapture(vid_source)
clip_len = 16
show_video = False
resize_size = (224, 224)
crop_size = (224, 224)

OUT_DIR = os.path.join('outputs')
os.makedirs(OUT_DIR, exist_ok=True)

transform = A.Compose([
    A.Resize(resize_size[1], resize_size[0]),
    A.Normalize(
        mean=[0.45, 0.45, 0.45],
        std=[0.225, 0.225, 0.225]
    )
])
print('Press q to quit')

model = mvit_v1_b(weights=None)
num_features = model.head[1].in_features
model.head[1] = nn.Linear(num_features, 12)
model.load_state_dict(torch.load('mvit-trained.pt', map_location=device))
model = model.to(device).eval()

with open('labels.csv', 'r') as f:
    class_names = f.readlines()
    f.close()

# Get the frame width and height.
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(5))
save_name = 'Output.mp4'.split('/')[-1].split('.')[0] # Output does not function with multiple inputs.
show_video = True

# Define codec and create VideoWriter object.
out = cv2.VideoWriter(
    f"{OUT_DIR}/{save_name}.mp4", 
    cv2.VideoWriter_fourcc(*'mp4v'), 
    fps, 
    (frame_width, frame_height)
)

time_trackers = {}
VIDEO_FPS = int(cap.get(cv2.CAP_PROP_FPS)) # Constant for fps of the video

for track_id in b_boxes:

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0) # Resets capture position to beginning for new tracked object
    frame_count = 0 #  Frame count for b_box index
    clips = [] # List to append and store the individual frames
    time_tracker = {} # Dictionary to store labels and respective durations
    
    while(cap.isOpened()):

        ret, frame = cap.read()
        b_box = b_boxes[track_id][frame_count] # Current bounding box

        if ret == True:

            xmin, ymin, xmax, ymax = b_box
            cropped_frame = frame[ymin:ymax, xmin:xmax]

            image = cropped_frame.copy()
            cropped_frame = cv2.cvtColor(cropped_frame, cv2.COLOR_BGR2RGB)
            cropped_frame = transform(image=cropped_frame)['image']
            clips.append(cropped_frame)

            if len(clips) == clip_len:
                input_frames = np.array(clips)
                input_frames = np.expand_dims(input_frames, axis=0) # Adds extra dimension
                input_frames = np.transpose(input_frames, (0, 4, 1, 2, 3)) # Transposes to get [1, 3, num_clips, height, width]

                # Convert the frames to tensor
                input_frames = torch.tensor(input_frames, dtype=torch.float32) 
                input_frames = input_frames.to(device)
                with torch.no_grad():
                    outputs = model(input_frames)
                
                _, preds = torch.max(outputs.data, 1) # Gets the prediction index
                label = class_names[preds].strip() # Maps predictions to the respective class names
                stripped_label = class_names[preds].strip().split(',')[-1].capitalize() # Reformats label for viewing
                frame_count += 1

                time_tracker[stripped_label] = time_tracker.get(stripped_label, 0) + 1 / VIDEO_FPS # Adds time & labels to dictionary

                cv2.putText(
                    image, 
                    f"{stripped_label}",
                    (15, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 
                    0.7, 
                    (0, 255, 255), 
                    1, 
                    lineType=cv2.LINE_AA
                )

                clips.pop(0)
                out.write(image)
        else:
            break

    time_trackers[int(track_id)] = time_tracker
    
cap.release()

Report

In [ ]:
with open("Test-Report.txt", "w") as f:
    sys.stdout = f
    for track_id, time_tracker in time_trackers.items():
        print(f"\nTotal length of each action for Student {track_id}:")
        for label, total_time in time_tracker.items():
            print(f"{label}: {total_time:.2f}s")

sys.stdout = sys.__stdout__